In [ ]:
import geopandas as gpd

cycling_network = gpd.read_file(r"output\cycling_network.gpkg", layer="edges")
intersections = gpd.read_file(r"output\cycling_network.gpkg", layer="nodes")

In [ ]:
cycling_network.head()

In [ ]:
intersections.head()

In [ ]:
import shapely
import osmnx as ox
import geopandas as gpd

def clip_to_espoo_kauniainen(gdf, buffer_m=500, crs="EPSG:3067"):
    
    places = [
        {"city": "Espoo", "country": "Finland"},
        {"city": "Kauniainen", "country": "Finland"},
    ]

    cities = ox.geocode_to_gdf(places)[["geometry"]].to_crs(crs)

    mask_geom = shapely.union_all(cities.geometry.values)
    mask = gpd.GeoDataFrame(geometry=[mask_geom], crs=crs)

    mask["geometry"] = mask.buffer(buffer_m)

    return gpd.clip(gdf.to_crs(crs), mask)


In [ ]:
cycling_network_espoo = clip_to_espoo_kauniainen(cycling_network)
intersections_espoo = clip_to_espoo_kauniainen(intersections)

In [ ]:
intersections_espoo.to_file("output/cycling_network.gpkg", layer='nodes_espoo', driver="GPKG")

In [ ]:
cycling_network_espoo.explore()

In [ ]:
cycling_network_espoo.columns

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import re

gdf = cycling_network_espoo.copy()

In [ ]:
def as_str_lower(val):
    if pd.isna(val):
        return ""
    return str(val).strip().lower()

def parse_maxspeed(val):
    if pd.isna(val):
        return np.nan
    s = str(val).lower().strip()

    # common non-numeric values
    if s in {"signals", "walk", "none", "variable"}:
        return np.nan

    # grab first number found
    match = re.search(r"(\d+)", s)
    if match:
        return float(match.group(1))
    return np.nan

def parse_lanes(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    match = re.search(r"(\d+)", s)
    if match:
        return float(match.group(1))
    return np.nan

for col in ["highway", "cycleway", "bicycle", "foot", "footway", "segregated",
            "surface", "smoothness", "lit", "motor_vehicle", "motorcar",
            "service", "sidewalk"]:
    if col in gdf.columns:
        gdf[col] = gdf[col].apply(as_str_lower)

gdf["maxspeed_num"] = gdf["maxspeed"].apply(parse_maxspeed) if "maxspeed" in gdf.columns else np.nan
gdf["lanes_num"] = gdf["lanes"].apply(parse_lanes) if "lanes" in gdf.columns else np.nan

In [ ]:
def has_protected_cycle_facility(row):
    highway = row["highway"]
    cycleway = row["cycleway"]
    motor_vehicle = row["motor_vehicle"]
    motorcar = row["motorcar"]

    # dedicated cycleway
    if highway == "cycleway":
        return True

    # cycle track / separated facility
    if cycleway in {"track", "opposite_track"}:
        return True

    # paths where cars are not allowed and bikes are allowed
    if highway in {"path", "footway", "pedestrian"}:
        if row["bicycle"] in {"yes", "designated", "permissive"} and motor_vehicle not in {"yes"} and motorcar not in {"yes"}:
            return True

    return False


def is_shared_path(row):
    highway = row["highway"]
    bicycle = row["bicycle"]
    foot = row["foot"]
    segregated = row["segregated"]

    if highway in {"path", "footway", "pedestrian"} and bicycle in {"yes", "designated", "permissive"}:
        return True

    if segregated in {"yes", "no"} and bicycle in {"yes", "designated", "permissive"} and foot in {"yes", "designated", "permissive"}:
        return True

    return False


def is_quiet_local_street(row):
    highway = row["highway"]
    maxspeed = row["maxspeed_num"]
    lanes = row["lanes_num"]
    motor_vehicle = row["motor_vehicle"]

    if highway == "living_street":
        return True

    if highway in {"residential", "service", "unclassified"}:
        speed_ok = pd.isna(maxspeed) or maxspeed <= 30
        lanes_ok = pd.isna(lanes) or lanes <= 2
        motor_ok = motor_vehicle not in {"yes"} or highway != "service"
        return speed_ok and lanes_ok and motor_ok

    return False


def is_busy_road(row):
    highway = row["highway"]
    maxspeed = row["maxspeed_num"]
    lanes = row["lanes_num"]
    cycleway = row["cycleway"]

    if highway in {"primary", "primary_link", "trunk", "trunk_link"}:
        return True

    if highway in {"secondary", "secondary_link"}:
        if cycleway not in {"track", "opposite_track"}:
            return True

    if highway in {"tertiary", "tertiary_link"}:
        if (not pd.isna(maxspeed) and maxspeed > 40) or (not pd.isna(lanes) and lanes >= 2):
            return True

    return False


def poor_surface_for_children(row):
    surface = row["surface"]
    smoothness = row["smoothness"]

    bad_surfaces = {
        "gravel", "ground", "dirt", "earth", "sand", "mud",
        "grass", "woodchips", "pebblestone", "cobblestone"
    }
    rough_smoothness = {"bad", "very_bad", "horrible", "very_horrible", "impassable"}

    return (surface in bad_surfaces) or (smoothness in rough_smoothness)

In [ ]:
def classify_child_cycling(row):
    highway = row["highway"]
    cycleway = row["cycleway"]
    maxspeed = row["maxspeed_num"]
    lanes = row["lanes_num"]
    lit = row["lit"]

    protected = has_protected_cycle_facility(row)
    shared = is_shared_path(row)
    quiet_local = is_quiet_local_street(row)
    busy = is_busy_road(row)
    rough = poor_surface_for_children(row)

    # Class 5: avoid
    if highway in {"motorway", "motorway_link", "trunk", "trunk_link"}:
        return 5, "avoid_not_child_friendly"

    if highway in {"primary", "primary_link"} and not protected:
        return 5, "avoid_not_child_friendly"

    # Class 1: very child-friendly
    if protected:
        if not rough:
            return 1, "very_child_friendly"
        else:
            return 2, "child_friendly_but_surface_limited"

    # Class 2: child-friendly with supervision
    if shared:
        if row["segregated"] == "yes" and not rough:
            return 1, "very_child_friendly"
        if not rough:
            return 2, "child_friendly_with_supervision"
        return 3, "rideable_but_low_comfort"

    if quiet_local:
        if not rough:
            return 2, "child_friendly_with_supervision"
        return 3, "rideable_but_low_comfort"

    # Class 4: stressful
    if busy:
        return 4, "stressful_not_child_friendly"

    # Class 3: doable but not ideal
    if highway in {"residential", "unclassified", "service", "tertiary", "tertiary_link"}:
        # slightly worse if speed or lanes are high
        if (not pd.isna(maxspeed) and maxspeed > 30) or (not pd.isna(lanes) and lanes > 2):
            return 4, "stressful_not_child_friendly"
        return 3, "rideable_but_low_comfort"

    # default fallback
    return 4, "stressful_not_child_friendly"

classified = gdf.apply(classify_child_cycling, axis=1, result_type="expand")
gdf[["child_class_num", "child_class_label"]] = classified

In [ ]:
def simplify_class(num):
    if num in {1, 2}:
        return "high_quality_for_children"
    elif num == 3:
        return "doable_but_not_ideal"
    else:
        return "not_child_friendly"

gdf["child_class_simple"] = gdf["child_class_num"].apply(simplify_class)

In [ ]:
gdf.to_file("output/cycling_network.gpkg", layer='edges_espoo', driver="GPKG")

In [ ]:
from shapely.ops import linemerge

gdf["orig_id"] = gdf["id"].astype(str)

group_cols = [
    "child_class_label",
    "highway",
    "cycleway",
    "maxspeed",
    "surface"
]

gdf[group_cols] = gdf[group_cols].fillna("unknown")

gdf_dissolved = gdf.dissolve(
    by=group_cols,
    aggfunc={
        "orig_id": lambda x: list(x),
        "length": "sum"
    }
).reset_index()

In [ ]:
gdf_dissolved

In [ ]:
gdf_dissolved.explore(
    column="child_class_label",
    categorical=True,
    legend=True,
    linewidth=4
)

In [ ]:
gdf_dissolved.explore(
    column="child_class_label",
    categorical=True,
    legend=True,
    linewidth=8
)

In [ ]:
import geopandas as gpd

cycling_network = gpd.read_file(r"output\cycling_network.gpkg", layer="edges_espoo")
intersections = gpd.read_file(r"output\cycling_network.gpkg", layer="nodes_espoo")

In [ ]:
intersections.head()

In [ ]:
dedicated_cycling = cycling_network[
    (cycling_network["segregated"] == "yes") &
    (cycling_network["highway"] == "cycleway")
]

dedicated_cycling.head()
len(dedicated_cycling)

In [ ]:
dedicated_cycling.explore()